In [24]:
!pip install -q tensorflow streamlit pyngrok

In [25]:
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [26]:
import tensorflow as tf

# Load image dataset
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

# Download and extract dataset
data_dir = tf.keras.utils.get_file(
    "flower_photos",
    origin=dataset_url,
    untar=True
)

print("Dataset downloaded successfully!")
print("Dataset location:", data_dir)

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step
Dataset downloaded successfully!
Dataset location: /root/.keras/datasets/flower_photos


In [27]:
# Image settings
img_height = 180
img_width = 180
batch_size = 32

# Create training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Create validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Get class names
class_names = train_ds.class_names

print("Classes:", class_names)
print("Number of classes:", len(class_names))


Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
Classes: ['flower_photos']
Number of classes: 1


In [28]:
# Fix the dataset path
import os

data_dir = os.path.join(data_dir, "flower_photos")

print("Correct dataset location:", data_dir)
print("Folders/classes:", os.listdir(data_dir))


Correct dataset location: /root/.keras/datasets/flower_photos/flower_photos
Folders/classes: ['daisy', 'LICENSE.txt', 'roses', 'dandelion', 'tulips', 'sunflowers']


In [29]:
# Image settings
img_height = 180
img_width = 180
batch_size = 32

# Create training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Create validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Get class names
class_names = train_ds.class_names

print("Classes:", class_names)
print("Number of classes:", len(class_names))


Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
Number of classes: 5


In [30]:
# Build a CNN model

model = tf.keras.Sequential([

    # Rescale pixel values
    tf.keras.layers.Rescaling(1./255),

    # Convolution layers
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    # Convert feature maps into one-dimensional data
    tf.keras.layers.Flatten(),

    # Fully connected layer
    tf.keras.layers.Dense(128, activation='relu'),

    # Output layer
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model structure
model.build((None, 180, 180, 3))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 178, 178, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 41, 41, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 51200)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     6,553,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,647,621 (25.36 MB)

 Trainable params: 6,647,621 (25.36 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
# Train the AI model

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 18s 117ms/step - accuracy: 0.4394 - loss: 1.3286 - val_accuracy: 0.5559 - val_loss: 1.1120
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.5906 - loss: 1.0154 - val_accuracy: 0.6267 - val_loss: 0.9628
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.6628 - loss: 0.8700 - val_accuracy: 0.6390 - val_loss: 0.9310
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.7592 - loss: 0.6557 - val_accuracy: 0.6526 - val_loss: 0.9496
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.8491 - loss: 0.4269 - val_accuracy: 0.6526 - val_loss: 0.9881


In [32]:
# Evaluate the model

loss, accuracy = model.evaluate(val_ds)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.6526 - loss: 0.9881
Validation Loss: 0.9881099462509155
Validation Accuracy: 0.6525885462760925


In [33]:
# Save the trained model

model.save("flower_classifier_model.keras")

print("Model saved successfully!")


Model saved successfully!


In [34]:
# Save class names

import json

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("Class names saved successfully!")
print(class_names)

Class names saved successfully!
['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [35]:
# Function to predict an image

import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

def predict_image(image_path):
    # Load image
    img = load_img(
        image_path,
        target_size=(180, 180)
    )

    # Convert image to array
    img_array = img_to_array(img)

    # Add batch dimension
    img_array = tf.expand_dims(img_array, 0)

    # Make prediction
    predictions = model.predict(img_array)

    # Get predicted class
    predicted_index = np.argmax(predictions[0])

    # Get confidence
    confidence = 100 * np.max(predictions[0])

    predicted_class = class_names[predicted_index]

    print("Predicted Class:", predicted_class)
    print("Confidence:", round(confidence, 2), "%")

    return predicted_class, confidence

In [36]:
from google.colab import files

# Upload an image
uploaded = files.upload()

# Get uploaded file name
image_path = list(uploaded.keys())[0]

# Predict image
predicted_class, confidence = predict_image(image_path)


Saving Screenshot 2026-07-29 161113.png to Screenshot 2026-07-29 161113.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 713ms/step
Predicted Class: roses
Confidence: 69.63 %


In [37]:
# Create Streamlit application
# Create corrected Streamlit application

app_code = '''
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

# Load trained model
model = tf.keras.models.load_model("flower_classifier_model.keras")

# Class names
class_names = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

# App title
st.title("🦷 Flower Classification using AI")
st.write("Upload an image to get an AI-based prediction.")

# Upload image
uploaded_file = st.file_uploader(
    "Choose an image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    # Open uploaded image
    image = Image.open(uploaded_file).convert("RGB")

    # Display image
    st.image(
        image,
        caption="Uploaded Image",
        use_container_width=True
    )

    # Resize image
    image = image.resize((180, 180))

    # Convert image to array
    img_array = np.array(image)

    # Add batch dimension
    img_array = tf.expand_dims(img_array, 0)

    # Prediction
    predictions = model.predict(img_array)

    # Get predicted class
    predicted_index = np.argmax(predictions[0])

    predicted_class = class_names[predicted_index]

    # Get confidence
    confidence = 100 * np.max(predictions[0])

    # Display results
    st.success("Prediction completed!")

    st.write("### Predicted Class:", predicted_class)
    st.write("### Confidence:", f"{confidence:.2f}%")
'''

# Save corrected app
with open("app.py", "w") as f:
    f.write(app_code)

print("Corrected Streamlit app created successfully!")


Corrected Streamlit app created successfully!


In [38]:
!pkill -f "streamlit run" || true

^C


In [39]:
!streamlit run app.py &>/content/logs.txt &

In [40]:
!pip install pyngrok


In [41]:
from pyngrok import ngrok
print("pyngrok installed successfully!")

pyngrok installed successfully!


In [42]:
# (ngrok auth token removed)


In [43]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("Open this URL in your browser:")
print(public_url)

Open this URL in your browser:
NgrokTunnel: "https://decorator-babble-durably.ngrok-free.dev" -> "http://localhost:8501"


In [44]:
!pip install streamlit pyngrok

In [45]:
!streamlit --version

Streamlit, version 1.60.0


In [46]:
!streamlit run app.py --server.port 8501 &



2026-07-29 20:19:33.366 Port 8501 is not available


In [47]:
!ls -l

total 78232
-rw-r--r-- 1 root root     1350 Jul 29 20:19  app.py
-rw-r--r-- 1 root root       55 Jul 29 20:17  class_names.json
-rw-r--r-- 1 root root      317 Jul 29 20:19  logs.txt
-rw-r--r-- 1 root root 79819068 Jul 29 20:17  oral_disease_detection_model.keras
drwxr-xr-x 1 root root     4096 Jun  4 13:39  sample_data
-rw-r--r-- 1 root root   270075 Jul 29 20:19 'Screenshot 2026-07-29 161113.png'


In [48]:
!python app.py

2026-07-29 20:19:40.908391: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1785356380.909870   10800 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11585 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-07-29 20:19:42.309 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-29 20:19:42.552 
  command:

    streamlit run app.py [ARGUMENTS]
2026-07-29 20:19:42.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-29 20:19:42.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-29 20:19:42.552 Thread 'MainThread': m

In [49]:
!cat logs.txt



2026-07-29 20:19:15.446 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.126.88.169:8501



In [50]:
!streamlit run app.py --server.port 8501



2026-07-29 20:21:23.205 Port 8501 is not available


In [51]:
from pyngrok import ngrok

# Close any old tunnels
ngrok.kill()

# Create a new tunnel
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://decorator-babble-durably.ngrok-free.dev" -> "http://localhost:8501"
